# No-Show Pipeline Factory & Preprocessing

Dit Jupyter Notebook bevat de implementatie van de `PipelineFactory` en de custom `GapRiskTransformer` voor het voorspellen van no-shows. 

### Rationale voor de Pipeline Volgorde
1. **ColumnTransformer (Eerste stap)**: We verwerken features op basis van hun type parallel om datalekken te voorkomen:
   - **Numeriek (`age`, `sessions_completed`)** -> Gestaafd met `StandardScaler` (nodig voor snellere convergentie van gradient descent).
   - **Categorisch (`treatment_type`)** -> Gecodeerd met `OneHotEncoder(handle_unknown='ignore')` (voorkomt crashes bij onbekende categorieën in productie).
   - **Custom feature (`last_session_gap`)** -> Binariseert het risico (>14 dagen) met onze `GapRiskTransformer`.
2. **Classifier (Tweede stap)**: De preprocessor stappen zijn direct verbonden aan de `LogisticRegression` classifier om een naadloze `.fit()` en `.predict()` stroom te garanderen.

## 1. Imports en Definities van de Classes

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn import set_config

class GapRiskTransformer(BaseEstimator, TransformerMixin):
    """
    Custom Scikit-Learn Transformer die de numerieke feature 'last_session_gap'
    omzet in een binaire feature 'high_risk' (1 als gap > 14 dagen, anders 0).
    """
    def __init__(self, threshold=14):
        self.threshold = threshold
        
    def fit(self, X, y=None):
        return self
        
    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            X_input = X.values
        else:
            X_input = X
            
        if len(X_input.shape) == 1:
            X_input = X_input.reshape(-1, 1)
            
        binary_risk = (X_input > self.threshold).astype(int)
        return binary_risk

    def get_feature_names_out(self, input_features=None):
        return np.array(['high_risk'], dtype=object)

class PipelineFactory:
    """
    Factory klasse voor het genereren van een robuuste scikit-learn
    pipeline voor no-show voorspellingen.
    """
    @staticmethod
    def create_pipeline(classifier=None):
        if classifier is None:
            classifier = LogisticRegression(random_state=42)
            
        numeric_features = ['age', 'sessions_completed']
        categorical_features = ['treatment_type']
        gap_feature = ['last_session_gap']
        
        numeric_transformer = Pipeline(steps=[
            ('scaler', StandardScaler())
        ])
        
        categorical_transformer = Pipeline(steps=[
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ])
        
        gap_transformer = Pipeline(steps=[
            ('risk_binarizer', GapRiskTransformer(threshold=14))
        ])
        
        preprocessor = ColumnTransformer(
            transformers=[
                ('num', numeric_transformer, numeric_features),
                ('cat', categorical_transformer, categorical_features),
                ('gap', gap_transformer, gap_feature)
            ],
            remainder='drop'
        )
        
        full_pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('classifier', classifier)
        ])
        
        return full_pipeline

## 2. Configureren en Fitten van de Pipeline

We schakelen de interactieve diagram-weergave in en trainen de pipeline op de synthetische patiëntendata.

In [ ]:
# Schakel interactieve diagramweergave in
set_config(display="diagram")

# Laad de data (zorg dat 'synthetic_patients.csv' in dezelfde map staat)
csv_path = 'synthetic_patients.csv'
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    X = df[['age', 'sessions_completed', 'treatment_type', 'last_session_gap']]
    y = df['no_show']
    
    # Pipeline genereren
    pipeline = PipelineFactory.create_pipeline()
    
    # Stel in dat preprocessors Pandas DataFrames moeten retourneren
    pipeline.set_output(transform="pandas")
    
    # Fitten
    pipeline.fit(X, y)
    print("Model succesvol gefit!")
else:
    print(f"Bestand '{csv_path}' niet gevonden. Run eerst 'generate_synthetic_data.py'.")

# Toon de pipeline (dit rendert het interactieve diagram in Jupyter)
pipeline

## 3. Testen met een Hypothetische Patiënt

We maken een proefvoorspelling voor een patiënt die een gat van 18 dagen (>14 dagen drempel) heeft.

In [ ]:
test_patient = pd.DataFrame([{
    'age': 34,
    'sessions_completed': 5,
    'treatment_type': 'depressie',
    'last_session_gap': 18
}])

# Voorspel klasse en kans
prediction = pipeline.predict(test_patient)
prob = pipeline.predict_proba(test_patient)[0][1]

print(f"Voorspelde uitkomst: {'No-Show' if prediction[0] == 1 else 'Aanwezig'}")
print(f"Berekende kans op No-Show: {prob:.2%}")

## 4. Inspecteren van de Getransformeerde Features (Pandas Output)

We kunnen de getransformeerde features rechtstreeks als een DataFrame bekijken om de werking van de encoders en scalers te controleren.

In [ ]:
# Haal de preprocessor los uit de getrainde pipeline
preprocessor = pipeline.named_steps['preprocessor']

# Transformeer de test_patient features
preprocessed_features = preprocessor.transform(test_patient)

# Toon het resultaat als Pandas DataFrame
preprocessed_features